# 03주차 실습: Spark RDD·DataFrame·I/O

강의: [PDF](../pdf/week03_apache_spark_basics.pdf) · **PDF 실제 페이지 6–15** (인쇄 번호와 다를 수 있음)

## 학습 목표
지연 실행과 DAG를 확인하고 명시적 스키마로 원천 데이터를 구조화한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## Spark 시작과 RDD

In [ ]:
from labkit.common import spark_session, event_schema
from pyspark.sql import functions as F
spark = spark_session('week03')
rdd = spark.sparkContext.parallelize(['spark data', 'spark sql', 'data lake'], 2)
counts = rdd.flatMap(lambda s: s.split()).map(lambda s: (s, 1)).reduceByKey(lambda a, b: a + b)
print(counts.toDebugString().decode())  # action 이전에는 lineage만 구성
assert dict(counts.collect()) == {'spark': 2, 'data': 2, 'sql': 1, 'lake': 1}

## 스키마와 집계 계획

In [ ]:
df = spark.read.schema(event_schema()).json(str(DATA / 'events.jsonl'))
df.printSchema()
agg = df.groupBy('product_id').agg(F.count('*').alias('events'), F.sum('value').alias('value'))
agg.explain(mode='formatted')
agg.show()
assert agg.agg(F.sum('events')).first()[0] == df.count()
out = output_dir('week03')
df.write.mode('overwrite').parquet(str(out / 'bronze'))
assert spark.read.parquet(str(out / 'bronze')).count() == 2400
spark.stop()

## 확장 과제 및 제출
select/filter/join/groupBy를 각각 추가하고 narrow/wide 변환을 구분하세요. scripts/00_fetch_to_landing.py와 10_bronze_batch.py를 실행해 원본·스키마·저장 포맷의 차이를 설명하세요.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.